In [4]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from yellowbrick.cluster import KElbowVisualizer
import matplotlib.pyplot as plt
import pandas as pd 
import seaborn as sns
from sksurv.base import SurvivalAnalysisMixin as s
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sksurv.preprocessing import encode_categorical
from sksurv.datasets import load_gbsg2
from sksurv.functions import StepFunction
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.ensemble import (ComponentwiseGradientBoostingSurvivalAnalysis, 
                            RandomSurvivalForest, 
                            ExtraSurvivalTrees, 
                            GradientBoostingSurvivalAnalysis, 
                            ExtraSurvivalTrees)
from sksurv.meta import EnsembleSelection, EnsembleSelectionRegressor
from sksurv.metrics import integrated_brier_score
from matplotlib.colors import ListedColormap
from mlxtend.evaluate import paired_ttest_5x2cv
from mlxtend.evaluate import combined_ftest_5x2cv
from lifelines import KaplanMeierFitter
from scipy.cluster import hierarchy
from lifelines.statistics import logrank_test, multivariate_logrank_test, pairwise_logrank_test
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, KFold
from lifelines.plotting import add_at_risk_counts
import scipy.stats
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna
from sklearn.model_selection import cross_val_score
from sksurv.metrics import integrated_brier_score
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor 
import statsmodels.api as sm
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
 
from sklearn.preprocessing import PowerTransformer


In [5]:
# OUS: Train data
OUS_D1 = pd.read_csv('OUS_D1.csv')
OUS_D2 = pd.read_csv('OUS_D2.csv')
OUS_D3 = pd.read_csv('OUS_D3.csv')
OUS_DFS_target = pd.read_csv('OUS_DFS_target.csv')
OUS_OS_target = pd.read_csv('OUS_OS_target.csv')
response_OUS = pd.read_csv('response_ous.csv', sep=';')

# MAASTRO: Test data 
MAASTRO_D1 = pd.read_csv('MAASTRO_D1.csv')
MAASTRO_D2 = pd.read_csv('MAASTRO_D2.csv')
MAASTRO_D3 = pd.read_csv('MAASTRO_D3.csv')
MAASTRO_DFS_target = pd.read_csv('MAASTRO_DFS_target.csv')
MAASTRO_OS_target = pd.read_csv('MAASTRO_OS_target.csv')
response_MAASTRO = pd.read_csv('maastro_response_full.csv', sep=',')

In [6]:
# Need to choose patient_id from OUS_D2 in response_OUS
data = list(OUS_D2['patient_id'])
mask = response_OUS['patient_id'].isin(data)
response_OUS = response_OUS[mask] 

# Merge OUS_D2 with response_OUS
clinical_train = pd.merge(OUS_D2, response_OUS, on='patient_id', how='inner')
clinical_train = clinical_train.loc[:, ~clinical_train.columns.isin(['DFS', 'event_DFS', 'LRC', 'event_LRC'])]

In [7]:
# Drop patient_id column
clinical_train = clinical_train.drop('patient_id', axis=1)

In [8]:
# Check null values in D2 
clinical_train.isnull().sum().sum()

0

In [9]:
""" Takes too long time 
# Check collinearity for OUS data 
df = clinical_train

# Create a correlation matrix
correlation_matrix = df.corr()

plt.figure(figsize=(15, 12))

# Plot a heatmap
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)
plt.show()
""" 

" Takes too long time \n# Check collinearity for OUS data \ndf = clinical_train\n\n# Create a correlation matrix\ncorrelation_matrix = df.corr()\n\nplt.figure(figsize=(15, 12))\n\n# Plot a heatmap\nsns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)\nplt.show()\n"

## Test dataset: MAASTRO 

In [10]:
(MAASTRO_D2['patient_id'] == MAASTRO_OS_target['patient_id']).sum()

99

In [11]:
# Rename the column name of response_MAASTRO 
response_MAASTRO.rename(columns = {'Index' : 'patient_id'}, inplace = True)

In [12]:
# need to choose patient_id from MAASTRO_D2 in response_MAASTRO
data = list(MAASTRO_D2['patient_id'])
mask = response_MAASTRO['patient_id'].isin(data)
response_MAASTRO = response_MAASTRO[mask] 
response_MAASTRO

,patient_id,OS,OS_event,LRC,LRC_event,DFS,DFS_event
0,1,62.43,0.0,62.43,0.0,62.43,0.0
1,2,60.00,0.0,60.00,0.0,60.00,0.0
2,3,44.43,1.0,8.83,1.0,8.83,1.0
3,4,37.20,1.0,19.37,0.0,19.73,1.0
5,6,59.23,0.0,59.23,0.0,59.23,0.0
...,...,...,...,...,...,...,...
109,110,19.00,1.0,13.27,1.0,13.27,1.0
110,111,85.87,1.0,82.83,0.0,85.87,1.0
111,112,42.87,0.0,42.87,0.0,42.87,0.0
112,113,58.93,0.0,58.93,0.0,58.93,0.0


In [13]:
# Merge MAASTRO_D2 with response_MAASTRO
clinical_test = pd.merge(MAASTRO_D2, response_MAASTRO, on='patient_id', how='inner')
clinical_test = clinical_test.loc[:, ~clinical_test.columns.isin(['DFS', 'DFS_event', 'LRC', 'LRC_event'])]
clinical_test

,patient_id,shape_Elongation,shape_Flatness,shape_LeastAxisLength,shape_MajorAxisLength,shape_Maximum2DDiameterColumn,shape_Maximum2DDiameterRow,shape_Maximum2DDiameterSlice,shape_Maximum3DDiameter,shape_MeshVolume,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,OS,OS_event
0,1,0.765178,0.610062,30.504395,50.002093,52.886671,58.000000,47.759816,58.864251,38067.208333,...,0.028808,0.837387,0.000026,0.001705,0.122209,0.000026,0.008291,0.000341,62.43,0.0
1,2,0.776540,0.504616,21.069386,41.753334,44.922155,44.294469,44.147480,48.723711,17870.791667,...,0.049615,0.806842,0.000167,0.002958,0.128976,0.000167,0.008148,0.000335,60.00,0.0
2,3,0.697164,0.478604,21.238275,44.375483,48.466483,45.891176,37.656341,48.969378,17426.500000,...,0.019514,0.830272,0.000057,0.001831,0.137282,0.000000,0.008641,0.000229,44.43,1.0
3,4,0.574636,0.446059,20.570459,46.115989,42.544095,47.507894,33.837849,55.226805,12463.791667,...,0.047155,0.760949,0.000000,0.003597,0.171595,0.000080,0.013187,0.000240,37.20,1.0
4,6,0.633419,0.480378,26.130155,54.394967,66.030296,56.320511,45.398238,67.089492,28300.125000,...,0.033990,0.824865,0.000000,0.002116,0.128134,0.000035,0.009379,0.000529,59.23,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,110,0.882411,0.577884,19.774388,34.218615,41.785165,40.261644,35.805028,43.520110,12822.458333,...,0.017489,0.834590,0.000000,0.001321,0.137194,0.000078,0.008706,0.000078,19.00,1.0
95,111,0.535802,0.455642,23.259099,51.046869,51.264022,51.264022,33.600595,52.440442,18368.291667,...,0.029979,0.821431,0.000000,0.000543,0.137620,0.000054,0.008907,0.000489,85.87,1.0
96,112,0.716610,0.631485,31.838169,50.417953,46.324939,56.035703,57.070132,57.671483,35384.541667,...,0.017354,0.859957,0.000000,0.000988,0.115099,0.000028,0.005700,0.000141,42.87,0.0
97,113,0.665145,0.628338,28.213278,44.901412,50.289164,50.596443,35.777088,51.478151,26180.208333,...,0.025399,0.847908,0.000000,0.001487,0.117654,0.000000,0.005759,0.000038,58.93,0.0


In [14]:
# Drop patient_id column
clinical_test = clinical_test.drop('patient_id', axis=1)

In [15]:
# Some rows have null values in OS, OS_event -> Remove those rows
clinical_test[clinical_test.isnull().any(axis=1)]
clinical_test = clinical_test.dropna(how='any',axis=0) 

,shape_Elongation,shape_Flatness,shape_LeastAxisLength,shape_MajorAxisLength,shape_Maximum2DDiameterColumn,shape_Maximum2DDiameterRow,shape_Maximum2DDiameterSlice,shape_Maximum3DDiameter,shape_MeshVolume,shape_MinorAxisLength,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,OS,OS_event


In [16]:
# X
X = clinical_train.loc[:, ~clinical_train.columns.isin(['OS', 'event_OS'])]

# y 
y = clinical_train.loc[:, ['OS', 'event_OS']]

# Set lower, upper time point and times for IBS calculation later 
lower, upper = np.percentile(y['OS'], [10, 90])
times = np.arange(lower, upper)

In [17]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [18]:
# Shape
print('X_train: ', X.shape)
print('y_train: ', y.shape)

X_train:  (139, 374)
y_train:  (139,)


In [19]:
clinical_test.rename(columns = {'OS_event' : 'event_OS'}, inplace = True)

In [20]:
# X
X_MAASTRO = clinical_test.loc[:, ~clinical_test.columns.isin(['OS', 'event_OS'])]

# y y_MAASTRO
y_MAASTRO = clinical_test.loc[:, ['OS', 'event_OS']]
lower, upper = np.percentile(y_MAASTRO['OS'], [10, 90])
times = np.arange(lower, upper)

# y into array 
lists = [] 
for i, j in zip(y_MAASTRO['event_OS'], y_MAASTRO['OS']): 
    lists.append((i, j))

y_MAASTRO = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

clinical_test.shape

(99, 376)

# Feature Selection

In [21]:
# Choose features from the result of Cox PLSR in R
selected_features = [
"first_order_Maximum_CT",                            
"LBP_201_PET",                              
"LBP_102_PET",                                
"gldm_SmallDependenceLowGrayLevelEmphasis_d_1_PET_b2",
"gldm_SmallDependenceLowGrayLevelEmphasis_d_1_CT_c16",  
"LBP_003_PET",                 
"gldm_SmallDependenceLowGrayLevelEmphasis_d_1_PET_c04"         
]

In [22]:
X_plsr = X.loc[:, selected_features]
X_new = X_plsr.copy()

In [23]:
# Selecct the columns from X_MAASTRO
MAASTRO_new = X_MAASTRO.loc[:, selected_features]

# Yeo-Johnson Transformation

In [24]:
# Copy the original X for later 
original_X = X.copy()

In [25]:
# Transform X_new 
# Set the categorical_columns
categorical_columns = ['female', 
                        'cavum_oris',
                        'oropharynx',
                        'hypopharynx',
                        'larynx',
                        'histgrade_high',
                        'hpv_related',
                        'charlson',
                        'uicc8_III-IV']

# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in X_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
X_new_categoric = X_new[columns_to_drop]
X_new_numeric = X_new.drop(columns=columns_to_drop, inplace=False)

# Apply Yeo-Johnson transformation
pt = PowerTransformer(method='yeo-johnson')
X_new_numeric_transformed = pt.fit_transform(X_new_numeric)

# Create DataFrame with transformed numerical data
X_new_numeric_transformed = pd.DataFrame(X_new_numeric_transformed, 
                                         columns=X_new_numeric.columns, 
                                         index=X_new.index)

# Concatenate transformed numerical data with categorical data
X_new_std = pd.concat([X_new_numeric_transformed, X_new_categoric], axis=1)

# Standardize X_MAASTRO 
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in MAASTRO_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
MAASTRO_new_categoric = MAASTRO_new[columns_to_drop]
MAASTRO_new_numeric = MAASTRO_new.drop(columns=columns_to_drop, inplace=False)

# Do the transformation for the numeric part 
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_index = MAASTRO_new_numeric.index 
MAASTRO_new_numeric_std = pt.transform(MAASTRO_new_numeric)
MAASTRO_new_numeric_std = pd.DataFrame(MAASTRO_new_numeric_std,
                                 columns=MAASTRO_new_numeric_columns, 
                                 index=MAASTRO_new_numeric_index)
MAASTRO_new_std = pd.concat([MAASTRO_new_numeric_std, MAASTRO_new_categoric], axis=1)

# Change the order of the X_new_std 
MAASTRO_new_std = MAASTRO_new_std[MAASTRO_new.columns]

In [26]:
X_new

,first_order_Maximum_CT,LBP_201_PET,LBP_102_PET,gldm_SmallDependenceLowGrayLevelEmphasis_d_1_PET_b2,gldm_SmallDependenceLowGrayLevelEmphasis_d_1_CT_c16,LBP_003_PET,gldm_SmallDependenceLowGrayLevelEmphasis_d_1_PET_c04
0,1763.283818,0.000062,0.000000,0.000959,0.000416,0.000123,0.001395
1,1432.109636,0.000349,0.000000,0.002776,0.001753,0.000349,0.002345
2,1685.931373,0.000000,0.000034,0.001179,0.000230,0.000000,0.001105
3,1329.347515,0.000000,0.000000,0.002748,0.000711,0.000300,0.002088
4,1197.225910,0.000399,0.000199,0.002309,0.001033,0.000000,0.002042
...,...,...,...,...,...,...,...
134,1267.959716,0.000000,0.000000,0.001347,0.000226,0.000000,0.001878
135,1788.278096,0.000079,0.000000,0.000850,0.000183,0.000039,0.000770
136,1478.297861,0.000000,0.000000,0.001111,0.000621,0.000000,0.001743
137,1207.068493,0.000054,0.000000,0.000935,0.000265,0.000000,0.001283


In [27]:
X_new_std

,first_order_Maximum_CT,LBP_201_PET,LBP_102_PET,gldm_SmallDependenceLowGrayLevelEmphasis_d_1_PET_b2,gldm_SmallDependenceLowGrayLevelEmphasis_d_1_CT_c16,LBP_003_PET,gldm_SmallDependenceLowGrayLevelEmphasis_d_1_PET_c04
0,1.022943,-0.074611,-0.784694,-0.456825,-0.128003,1.042615,-0.301870
1,0.072727,1.689445,-0.784694,1.829690,1.761655,1.929439,1.172866
2,0.853782,-1.065926,0.236612,0.033412,-0.861897,-0.799521,-1.042887
3,-0.390532,-1.065926,-0.784694,1.814149,0.668885,1.846691,0.881234
4,-1.187658,1.782787,1.784568,1.522029,1.208077,-0.799521,0.822455
...,...,...,...,...,...,...,...
134,-0.727488,-1.065926,-0.784694,0.352487,-0.877225,-0.799521,0.592895
135,1.072604,0.138716,-0.784694,-0.735470,-1.082845,-0.000653,-2.156858
136,0.247868,-1.065926,-0.784694,-0.108155,0.464990,-0.799521,0.378812
137,-1.118344,-0.175482,-0.784694,-0.515193,-0.706059,-0.799521,-0.567395


In [28]:
MAASTRO_new 

,first_order_Maximum_CT,LBP_201_PET,LBP_102_PET,gldm_SmallDependenceLowGrayLevelEmphasis_d_1_PET_b2,gldm_SmallDependenceLowGrayLevelEmphasis_d_1_CT_c16,LBP_003_PET,gldm_SmallDependenceLowGrayLevelEmphasis_d_1_PET_c04
0,1226.795917,0.000026,0.000026,0.000768,0.000241,0.000026,0.001101
1,1876.570382,0.000167,0.000167,0.001213,0.000621,0.000056,0.001861
2,1269.217095,0.000000,0.000057,0.001176,0.000327,0.000286,0.001353
3,1951.924395,0.000080,0.000000,0.001192,0.001419,0.000160,0.001420
4,1646.853670,0.000035,0.000000,0.000766,0.000165,0.000071,0.001340
...,...,...,...,...,...,...,...
94,2311.845731,0.000078,0.000000,0.001137,0.000739,0.000000,0.001765
95,1299.857157,0.000054,0.000000,0.000962,0.001130,0.000109,0.001594
96,1256.024949,0.000028,0.000000,0.000621,0.000226,0.000000,0.001435
97,1668.742962,0.000000,0.000000,0.000947,0.000296,0.000114,0.001167


In [29]:
MAASTRO_new_std

,first_order_Maximum_CT,LBP_201_PET,LBP_102_PET,gldm_SmallDependenceLowGrayLevelEmphasis_d_1_PET_b2,gldm_SmallDependenceLowGrayLevelEmphasis_d_1_CT_c16,LBP_003_PET,gldm_SmallDependenceLowGrayLevelEmphasis_d_1_PET_c04
0,-0.984802,-0.597387,0.033184,-0.960377,-0.811666,-0.238970,-1.054082
1,1.231237,0.946684,1.682719,0.101598,0.464229,0.263801,0.566401
2,-0.720051,-1.065926,0.693043,0.027699,-0.453072,1.817599,-0.397642
3,1.348682,0.152800,-0.784694,0.060650,1.579512,1.314752,-0.245977
4,0.758335,-0.453142,-0.784694,-0.967045,-1.171859,0.471935,-0.428826
...,...,...,...,...,...,...,...
94,1.750433,0.126817,-0.784694,-0.052777,0.727222,-0.799521,0.414211
95,-0.545966,-0.173978,-0.784694,-0.450240,1.322893,0.907497,0.110883
96,-0.799285,-0.565019,-0.784694,-1.409205,-0.877027,-0.799521,-0.214415
97,0.812692,-1.065926,-0.784694,-0.485449,-0.577241,0.962590,-0.869387


# Modelling 

### 1. CoxPHSurvivalAnalysis

#### Train

In [30]:
# Setting the y format for skf below  
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class()
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxPHSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxPHSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-16 01:30:44,582] A new study created in memory with name: no-name-72da8fdb-0417-4904-b0cb-76e6e5887dea


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.6277056277056277
Fold 2 C-index: 0.7053571428571429
Fold 3 C-index: 0.6764705882352942
Fold 4 C-index: 0.6118143459915611


[I 2024-04-16 01:31:04,373] A new study created in memory with name: no-name-48b13c95-2cea-4ab5-87fd-47cfff6279ed


Fold 5 C-index: 0.6572769953051644
[I 2024-04-16 01:31:04,215] Trial 0 finished with value: 0.655724940018958 and parameters: {}. Best is trial 0 with value: 0.655724940018958.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.655724940018958], datetime_start=datetime.datetime(2024, 4, 16, 1, 30, 44, 721169), datetime_complete=datetime.datetime(2024, 4, 16, 1, 31, 4, 213597), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.655724940018958


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.1969409117307865
Fold 2 IBS: 0.19322839929322455
Fold 3 IBS: 0.18516741534279244
Fold 4 IBS: 0.21493247358260703
Fold 5 IBS: 0.2410128885614412
[I 2024-04-16 01:31:06,938] Trial 0 finished with value: 0.20625641770217035 and parameters: {}. Best is trial 0 with value: 0.20625641770217035.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.20625641770217035], datetime_start=datetime.datetime(2024, 4, 16, 1, 31, 4, 739152), datetime_complete=datetime.datetime(2024, 4, 16, 1, 31, 6, 937766), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.20625641770217035


In [31]:
# Setting a dictionary to save the train results 
train_cindex = {} 
train_ibs = {} 

# Saving the values to the dictionary 
train_cindex['CoxPH'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxPH'] = np.round(study_ibs.best_value, 3)

In [32]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.656
train_ibs:  0.206


#### Test

In [33]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [34]:
# Test on MAASTRO 
cph = CoxPHSurvivalAnalysis()

cph.fit(X_new_std, y)

# Save C-index 
c_index = cph.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print('Concordance index:', c_index)

# Save IBS 
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in cph.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print('IBS score:', ibs)

CoxPHSurvivalAnalysis()

Concordance index: 0.591
IBS score: 0.238


In [35]:
# Setting a dictionary to save the test results 
test_cindex = {} 
test_ibs = {} 

In [36]:
# Saving the values to the dictionary 
test_cindex['CoxPH'] = c_index
test_ibs['CoxPH'] = ibs

### 2. CoxnetSurvivalAnalysis

#### Train

In [37]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=0.0000001, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 01:31:07,541] A new study created in memory with name: no-name-7835e9c1-e3c5-453e-9ceb-6cf5ffe68947


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.5952380952380952
Fold 2 C-index: 0.7321428571428571
Fold 3 C-index: 0.7009803921568627


[I 2024-04-16 01:31:08,855] A new study created in memory with name: no-name-2787e164-89cb-4cda-8116-08cbebbd03fd


Fold 4 C-index: 0.70042194092827
Fold 5 C-index: 0.647887323943662
[I 2024-04-16 01:31:08,840] Trial 0 finished with value: 0.6753341218819494 and parameters: {}. Best is trial 0 with value: 0.6753341218819494.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6753341218819494], datetime_start=datetime.datetime(2024, 4, 16, 1, 31, 7, 608955), datetime_complete=datetime.datetime(2024, 4, 16, 1, 31, 8, 840070), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6753341218819494


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.21397651842800988
Fold 2 IBS: 0.2215779102658637
Fold 3 IBS: 0.20453594230284272
Fold 4 IBS: 0.22473803637388992
Fold 5 IBS: 0.2181243145661695
[I 2024-04-16 01:31:10,143] Trial 0 finished with value: 0.21659054438735512 and parameters: {}. Best is trial 0 with value: 0.21659054438735512.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.21659054438735512], datetime_start=datetime.datetime(2024, 4, 16, 1, 31, 8, 948603), datetime_complete=datetime.datetime(2024, 4, 16, 1, 31, 10, 143210), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.21659054438735512


In [38]:
train_cindex['CoxRidge'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxRidge'] = np.round(study_ibs.best_value, 3)

In [39]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.675
train_ibs:  0.217


#### Test

In [40]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [41]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 0.0000001
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_cindex : 0.616


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_ibs:  0.221


In [42]:
# Saving the values to the dictionary 
test_cindex['CoxRidge'] = c_index
test_ibs['CoxRidge'] = ibs

### 3. CoxnetSurvivalAnalysis - Lasso

#### Train

In [43]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=1, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 01:31:10,775] A new study created in memory with name: no-name-aa49804d-348c-4bda-97f4-c315f9e3e208


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.6277056277056277
Fold 2 C-index: 0.7008928571428571
Fold 3 C-index: 0.6813725490196079
Fold 4 C-index: 0.6075949367088608


[I 2024-04-16 01:31:12,919] A new study created in memory with name: no-name-e7a16b48-7e34-4d90-b650-a9ff5b11ee18


Fold 5 C-index: 0.6666666666666666
[I 2024-04-16 01:31:12,908] Trial 0 finished with value: 0.656846527448724 and parameters: {}. Best is trial 0 with value: 0.656846527448724.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.656846527448724], datetime_start=datetime.datetime(2024, 4, 16, 1, 31, 10, 854919), datetime_complete=datetime.datetime(2024, 4, 16, 1, 31, 12, 908363), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.656846527448724


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.19602106389596613
Fold 2 IBS: 0.19297396519554105
Fold 3 IBS: 0.18513419898365446
Fold 4 IBS: 0.2142933165670212
Fold 5 IBS: 0.24022040689201665
[I 2024-04-16 01:31:14,621] Trial 0 finished with value: 0.2057285903068399 and parameters: {}. Best is trial 0 with value: 0.2057285903068399.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.2057285903068399], datetime_start=datetime.datetime(2024, 4, 16, 1, 31, 12, 960465), datetime_complete=datetime.datetime(2024, 4, 16, 1, 31, 14, 620758), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.2057285903068399


In [44]:
train_cindex['CoxLasso'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxLasso'] = np.round(study_ibs.best_value, 3)

In [45]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.657
train_ibs:  0.206


#### Test

In [46]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [47]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 1
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_cindex : 0.59


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_ibs:  0.237


In [48]:
# Saving the values to the dictionary 
test_cindex['CoxLasso'] = c_index
test_ibs['CoxLasso'] = ibs

### 4. CoxnetSurvivalAnalysis - ElasticNet

#### Train

In [49]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        l1_ratio = trial.suggest_float("l1_ratio", 0.0001, 1)
        
        # Create and fit survival model 
        model = model_class(l1_ratio=l1_ratio, 
                           fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 01:31:15,429] A new study created in memory with name: no-name-5f29fad6-9da7-45d9-bc35-ff6d943f8a5e


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6320346320346321
Fold 2 C-index: 0.7008928571428571
Fold 3 C-index: 0.6813725490196079
Fold 4 C-index: 0.6075949367088608
Fold 5 C-index: 0.6666666666666666
[I 2024-04-16 01:31:16,903] Trial 0 finished with value: 0.6577123283145249 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.6577123283145249.
Fold 1 C-index: 0.6320346320346321
Fold 2 C-index: 0.7008928571428571
Fold 3 C-index: 0.6813725490196079
Fold 4 C-index: 0.6075949367088608
Fold 5 C-index: 0.6666666666666666
[I 2024-04-16 01:31:18,091] Trial 1 finished with value: 0.6577123283145249 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 0 with value: 0.6577123283145249.
Fold 1 C-index: 0.6277056277056277
Fold 2 C-index: 0.7008928571428571
Fold 3 C-index: 0.6813725490196079
Fold 4 C-index: 0.6075949367088608
Fold 5 C-index: 0.6666666666666666
[I 2024-04-16 01:31:19,510] Trial 2 finished with value: 0.656846527448724 and parameters: {'l1_ratio': 0.22692876841884668}. 

Fold 1 C-index: 0.6320346320346321
Fold 2 C-index: 0.7008928571428571
Fold 3 C-index: 0.6813725490196079
Fold 4 C-index: 0.6075949367088608
Fold 5 C-index: 0.6666666666666666
[I 2024-04-16 01:32:01,280] Trial 24 finished with value: 0.6577123283145249 and parameters: {'l1_ratio': 0.6776130739315861}. Best is trial 0 with value: 0.6577123283145249.
Fold 1 C-index: 0.6277056277056277
Fold 2 C-index: 0.7008928571428571
Fold 3 C-index: 0.6813725490196079
Fold 4 C-index: 0.6075949367088608
Fold 5 C-index: 0.6666666666666666
[I 2024-04-16 01:32:03,852] Trial 25 finished with value: 0.656846527448724 and parameters: {'l1_ratio': 0.8125286485497188}. Best is trial 0 with value: 0.6577123283145249.
Fold 1 C-index: 0.6320346320346321
Fold 2 C-index: 0.7008928571428571
Fold 3 C-index: 0.6813725490196079
Fold 4 C-index: 0.6075949367088608
Fold 5 C-index: 0.6666666666666666
[I 2024-04-16 01:32:07,169] Trial 26 finished with value: 0.6577123283145249 and parameters: {'l1_ratio': 0.5421167620714675}.

Fold 1 C-index: 0.6320346320346321
Fold 2 C-index: 0.7008928571428571
Fold 3 C-index: 0.6813725490196079
Fold 4 C-index: 0.6075949367088608
Fold 5 C-index: 0.6666666666666666
[I 2024-04-16 01:33:11,235] Trial 48 finished with value: 0.6577123283145249 and parameters: {'l1_ratio': 0.45330460100324665}. Best is trial 0 with value: 0.6577123283145249.
Fold 1 C-index: 0.6320346320346321
Fold 2 C-index: 0.7008928571428571
Fold 3 C-index: 0.6813725490196079
Fold 4 C-index: 0.6075949367088608
Fold 5 C-index: 0.6666666666666666
[I 2024-04-16 01:33:13,313] Trial 49 finished with value: 0.6577123283145249 and parameters: {'l1_ratio': 0.587848527895132}. Best is trial 0 with value: 0.6577123283145249.
Fold 1 C-index: 0.6320346320346321
Fold 2 C-index: 0.7008928571428571
Fold 3 C-index: 0.6813725490196079
Fold 4 C-index: 0.6075949367088608
Fold 5 C-index: 0.6666666666666666
[I 2024-04-16 01:33:15,270] Trial 50 finished with value: 0.6577123283145249 and parameters: {'l1_ratio': 0.5027665121167861}

...

Fold 1 IBS: 0.19607426324452154
Fold 2 IBS: 0.19291796526379473
Fold 3 IBS: 0.18513414094859856
Fold 4 IBS: 0.2142802941958011
Fold 5 IBS: 0.24013371045108928
[I 2024-04-16 01:35:04,995] Trial 0 finished with value: 0.20570807482076106 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.20570807482076106.
Fold 1 IBS: 0.196178160147061
Fold 2 IBS: 0.19276640490872884
Fold 3 IBS: 0.18513928059360607
Fold 4 IBS: 0.21431143617525175
Fold 5 IBS: 0.23981444334062385
[I 2024-04-16 01:35:07,336] Trial 1 finished with value: 0.2056419450330543 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 1 with value: 0.2056419450330543.
Fold 1 IBS: 0.19617043924369143
Fold 2 IBS: 0.1927536401284449
Fold 3 IBS: 0.1851412757034659
Fold 4 IBS: 0.2142832189636069
Fold 5 IBS: 0.23966506215978026
[I 2024-04-16 01:35:10,032] Trial 2 finished with value: 0.20560272723979792 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 2 with value: 0.20560272723979792

Fold 1 IBS: 0.19628000561567796
Fold 2 IBS: 0.19264591971385459
Fold 3 IBS: 0.18514833426583724
Fold 4 IBS: 0.21429629848935952
Fold 5 IBS: 0.23947026387589404
[I 2024-04-16 01:36:05,611] Trial 25 finished with value: 0.20556816439212464 and parameters: {'l1_ratio': 0.06766854141508033}. Best is trial 18 with value: 0.2055613706017152.
Fold 1 IBS: 0.21344889697661673
Fold 2 IBS: 0.22055908509104202
Fold 3 IBS: 0.2037910348514832
Fold 4 IBS: 0.22399916798848396
Fold 5 IBS: 0.2180296410983425
[I 2024-04-16 01:36:07,447] Trial 26 finished with value: 0.21596556520119367 and parameters: {'l1_ratio': 0.015423757551295547}. Best is trial 18 with value: 0.2055613706017152.
Fold 1 IBS: 0.19606397051451482
Fold 2 IBS: 0.19289942457205242
Fold 3 IBS: 0.18513444210503735
Fold 4 IBS: 0.21429747009648822
Fold 5 IBS: 0.23999853978081567
[I 2024-04-16 01:36:09,923] Trial 27 finished with value: 0.2056787694137817 and parameters: {'l1_ratio': 0.58416616744615}. Best is trial 18 with value: 0.205561370

Fold 5 IBS: 0.239531825454713
[I 2024-04-16 01:37:15,540] Trial 49 finished with value: 0.20557157815762012 and parameters: {'l1_ratio': 0.1254528423575577}. Best is trial 36 with value: 0.20554511524025393.
Fold 1 IBS: 0.19621316397714889
Fold 2 IBS: 0.19269257717510702
Fold 3 IBS: 0.1851422206421247
Fold 4 IBS: 0.21430059784505248
Fold 5 IBS: 0.2396107519746553
[I 2024-04-16 01:37:18,750] Trial 50 finished with value: 0.20559186232281768 and parameters: {'l1_ratio': 0.1784034233166914}. Best is trial 36 with value: 0.20554511524025393.
Fold 1 IBS: 0.21268742875213992
Fold 2 IBS: 0.21914754413479917
Fold 3 IBS: 0.18515228675934306
Fold 4 IBS: 0.2142901632238732
Fold 5 IBS: 0.23950700809034997
[I 2024-04-16 01:37:22,710] Trial 51 finished with value: 0.21415688619210105 and parameters: {'l1_ratio': 0.03940854054912912}. Best is trial 36 with value: 0.20554511524025393.
Fold 1 IBS: 0.19629214695819314
Fold 2 IBS: 0.1926219527261495
Fold 3 IBS: 0.1851516072514167
Fold 4 IBS: 0.2142907948

Fold 1 IBS: 0.196286739680928
Fold 2 IBS: 0.19265366490340924
Fold 3 IBS: 0.1851480871073592
Fold 4 IBS: 0.21430405096308916
Fold 5 IBS: 0.23949210283802028
[I 2024-04-16 01:38:35,511] Trial 74 finished with value: 0.20557692909856118 and parameters: {'l1_ratio': 0.06871556465927647}. Best is trial 36 with value: 0.20554511524025393.
Fold 1 IBS: 0.21302139910858334
Fold 2 IBS: 0.21975838021497485
Fold 3 IBS: 0.18515340128832472
Fold 4 IBS: 0.2234194295456223
Fold 5 IBS: 0.2394523242845926
[I 2024-04-16 01:38:37,205] Trial 75 finished with value: 0.21616098688841956 and parameters: {'l1_ratio': 0.028621971113182047}. Best is trial 36 with value: 0.20554511524025393.
Fold 1 IBS: 0.1962105325667388
Fold 2 IBS: 0.19265188880986528
Fold 3 IBS: 0.18514561150929243
Fold 4 IBS: 0.21431480274146802
Fold 5 IBS: 0.23949387268473116
[I 2024-04-16 01:38:39,057] Trial 76 finished with value: 0.20556334166241913 and parameters: {'l1_ratio': 0.11081271262960429}. Best is trial 36 with value: 0.2055451

Fold 4 IBS: 0.21430108421599545
Fold 5 IBS: 0.2401228436971971
[I 2024-04-16 01:39:33,705] Trial 98 finished with value: 0.2057018203682534 and parameters: {'l1_ratio': 0.7667338152719219}. Best is trial 36 with value: 0.20554511524025393.
Fold 1 IBS: 0.19623831345492668
Fold 2 IBS: 0.1926636410581049
Fold 3 IBS: 0.18514364915749978
Fold 4 IBS: 0.21430253945348898
Fold 5 IBS: 0.23965438244574724
[I 2024-04-16 01:39:35,676] Trial 99 finished with value: 0.2056005051139535 and parameters: {'l1_ratio': 0.13788271573671446}. Best is trial 36 with value: 0.20554511524025393.


* Best trial for IBS: 
 FrozenTrial(number=36, state=TrialState.COMPLETE, values=[0.20554511524025393], datetime_start=datetime.datetime(2024, 4, 16, 1, 36, 35, 754414), datetime_complete=datetime.datetime(2024, 4, 16, 1, 36, 38, 169171), params={'l1_ratio': 0.043525712831569995}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'l1_ratio': FloatDistribution(high=1.0, log=False, low=0.0001, step=

In [50]:
train_cindex['CoxElastic'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxElastic'] = np.round(study_ibs.best_value, 3)

In [51]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.658
train_ibs:  0.206


#### Test

In [52]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [53]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    return model_class(**best_params, fit_baseline_model=True)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.6964995386793018)

test_cindex : 0.59


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.043525712831569995)

test_ibs:  0.237


In [54]:
# Saving the values to the dictionary 
test_cindex['CoxElastic'] = c_index
test_ibs['CoxElastic'] = ibs

### 5. Random Survival Forest

#### Train

In [55]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None])
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics
        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score,
                            warm_start=warm_start,
                            max_depth=max_depth,
                            max_features=max_features,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_samples=max_samples, 
                            random_state=123)

        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])
            
            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(RandomSurvivalForest, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(RandomSurvivalForest, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-16 01:39:36,718] A new study created in memory with name: no-name-3b463807-d7aa-4eef-bc70-fa4fe2d90064


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5887445887445888
Fold 2 C-index: 0.6964285714285714
Fold 3 C-index: 0.6372549019607843
Fold 4 C-index: 0.609704641350211
Fold 5 C-index: 0.6854460093896714
[I 2024-04-16 01:39:53,788] Trial 0 finished with value: 0.6435157425747653 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'max_samples': 0.7163467647263769, 'max_features': None, 'min_weight_fraction_leaf': 0.2192861223398122, 'warm_start': False}. Best is trial 0 with value: 0.6435157425747653.
Fold 1 C-index: 0.6017316017316018
Fold 2 C-index: 0.6607142857142857
Fold 3 C-index: 0.6323529411764706
Fold 4 C-index: 0.6160337552742616
Fold 5 C-index: 0.6619718309859155
[I 2024-04-16 01:40:03,950] Trial 1 finished with value: 0.634560882976507 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 5, 'min_samples_leaf': 4, 'max_depth': 11, 'n_estimators': 266, 'oob_score': False, 'max_samples': 0.7520097923745717, 'm

Fold 1 C-index: 0.5800865800865801
Fold 2 C-index: 0.7321428571428571
Fold 3 C-index: 0.6029411764705882
Fold 4 C-index: 0.630801687763713
Fold 5 C-index: 0.6854460093896714
[I 2024-04-16 01:44:13,688] Trial 16 finished with value: 0.646283662170682 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 6, 'min_samples_leaf': 19, 'max_depth': 7, 'n_estimators': 364, 'oob_score': False, 'max_samples': 0.7515853579413085, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.12025117221320178, 'warm_start': False}. Best is trial 16 with value: 0.646283662170682.
Fold 1 C-index: 0.5714285714285714
Fold 2 C-index: 0.7321428571428571
Fold 3 C-index: 0.6029411764705882
Fold 4 C-index: 0.630801687763713
Fold 5 C-index: 0.6713615023474179
[I 2024-04-16 01:44:49,382] Trial 17 finished with value: 0.6417351590306295 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 5, 'min_samples_leaf': 20, 'max_depth': 7, 'n_estimators': 493, 'oob_score': False, 'max_samples': 0.8174822258864815

Fold 1 C-index: 0.5930735930735931
Fold 2 C-index: 0.8616071428571429
Fold 3 C-index: 0.6519607843137255
Fold 4 C-index: 0.7341772151898734
Fold 5 C-index: 0.7230046948356808
[I 2024-04-16 01:46:52,899] Trial 31 finished with value: 0.7127646860540031 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 13, 'min_samples_leaf': 14, 'max_depth': 5, 'n_estimators': 386, 'oob_score': True, 'max_samples': 0.8209031154082257, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.09599093645659847, 'warm_start': True}. Best is trial 23 with value: 0.7226788291857692.
Fold 1 C-index: 0.5887445887445888
Fold 2 C-index: 0.8616071428571429
Fold 3 C-index: 0.6568627450980392
Fold 4 C-index: 0.7088607594936709
Fold 5 C-index: 0.7323943661971831
[I 2024-04-16 01:47:02,092] Trial 32 finished with value: 0.7096939204781249 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 13, 'min_samples_leaf': 14, 'max_depth': 2, 'n_estimators': 284, 'oob_score': True, 'max_samples': 0.7868995414870

...

Fold 5 IBS: 0.21344899574141954
[I 2024-04-16 01:54:05,524] Trial 31 finished with value: 0.20469855403503315 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 14, 'min_samples_leaf': 2, 'max_depth': 18, 'n_estimators': 391, 'oob_score': True, 'max_samples': 0.8945361564514921, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.019207941499538344}. Best is trial 30 with value: 0.2040563796926822.
Fold 1 IBS: 0.19906422113284747
Fold 2 IBS: 0.206965012233294
Fold 3 IBS: 0.1928488518193958
Fold 4 IBS: 0.21332960673286458
Fold 5 IBS: 0.2178341182823558
[I 2024-04-16 01:54:22,910] Trial 32 finished with value: 0.20600836204015155 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 16, 'min_samples_leaf': 2, 'max_depth': 18, 'n_estimators': 464, 'oob_score': True, 'max_samples': 0.7842251850941289, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.0029357598861563597}. Best is trial 30 with value: 0.2040563796926822.
Fold 1 IBS: 0.20408617163679496
Fold 2 IBS: 0.205

Fold 1 IBS: 0.20104036243969675
Fold 2 IBS: 0.20536630217365237
Fold 3 IBS: 0.19037076567072828
Fold 4 IBS: 0.21572005633461494
Fold 5 IBS: 0.21011692141960442
[I 2024-04-16 01:57:21,742] Trial 47 finished with value: 0.20452288160765933 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 16, 'min_samples_leaf': 4, 'max_depth': 20, 'n_estimators': 340, 'oob_score': True, 'max_samples': 0.6977990144579234, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.025965903634064653}. Best is trial 44 with value: 0.20391475734721695.
Fold 1 IBS: 0.20728783436618
Fold 2 IBS: 0.2049967559036454
Fold 3 IBS: 0.19498736250827226
Fold 4 IBS: 0.2148072924530503
Fold 5 IBS: 0.21417413689935214
[I 2024-04-16 01:57:31,597] Trial 48 finished with value: 0.2072506764261 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 16, 'min_samples_leaf': 4, 'max_depth': 20, 'n_estimators': 347, 'oob_score': True, 'max_samples': 0.7057566829089937, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 

Fold 5 IBS: 0.20982282453299972
[I 2024-04-16 02:00:13,898] Trial 62 finished with value: 0.20327581178298254 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 15, 'min_samples_leaf': 3, 'max_depth': 20, 'n_estimators': 265, 'oob_score': True, 'max_samples': 0.6858370874030949, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.04150152613199446}. Best is trial 62 with value: 0.20327581178298254.
Fold 1 IBS: 0.20601281152111253
Fold 2 IBS: 0.20449414602624297
Fold 3 IBS: 0.1947762532232537
Fold 4 IBS: 0.21288218188391347
Fold 5 IBS: 0.20843731012181801
[I 2024-04-16 02:00:22,591] Trial 63 finished with value: 0.20532054055526813 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 14, 'min_samples_leaf': 5, 'max_depth': 19, 'n_estimators': 276, 'oob_score': True, 'max_samples': 0.6346530024976436, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.0365353414774943}. Best is trial 62 with value: 0.20327581178298254.
Fold 1 IBS: 0.2118991292296007
Fold 2 IBS: 0.20610

Fold 1 IBS: 0.21397762546898652
Fold 2 IBS: 0.22147883751600755
Fold 3 IBS: 0.20486980135357746
Fold 4 IBS: 0.22464994266370494
Fold 5 IBS: 0.21845929256270377
[I 2024-04-16 02:03:00,599] Trial 78 finished with value: 0.21668709991299603 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 8, 'min_samples_leaf': 5, 'max_depth': 19, 'n_estimators': 483, 'oob_score': True, 'max_samples': 0.5153355818628218, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.28987424398930645}. Best is trial 62 with value: 0.20327581178298254.
Fold 1 IBS: 0.21153493071166155
Fold 2 IBS: 0.20333834370429396
Fold 3 IBS: 0.19871181332894533
Fold 4 IBS: 0.21447173484121768
Fold 5 IBS: 0.21412319034910962
[I 2024-04-16 02:03:11,572] Trial 79 finished with value: 0.20843600258704562 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 11, 'min_samples_leaf': 3, 'max_depth': 17, 'n_estimators': 454, 'oob_score': True, 'max_samples': 0.5747714627136549, 'max_features': 'sqrt', 'min_weight_fraction_

Fold 5 IBS: 0.21386443466586402
[I 2024-04-16 02:05:38,316] Trial 93 finished with value: 0.2045595345822267 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 12, 'min_samples_leaf': 3, 'max_depth': 19, 'n_estimators': 414, 'oob_score': True, 'max_samples': 0.5498688861577421, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.029844645175168258}. Best is trial 85 with value: 0.20295984562926828.
Fold 1 IBS: 0.2095914999638493
Fold 2 IBS: 0.20254997001853908
Fold 3 IBS: 0.19693136056894167
Fold 4 IBS: 0.21511135614957475
Fold 5 IBS: 0.21430028358953207
[I 2024-04-16 02:05:49,271] Trial 94 finished with value: 0.20769689405808736 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 12, 'min_samples_leaf': 2, 'max_depth': 18, 'n_estimators': 445, 'oob_score': True, 'max_samples': 0.5904725395446608, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.07562078772066635}. Best is trial 85 with value: 0.20295984562926828.
Fold 1 IBS: 0.2099450942179432
Fold 2 IBS: 0.2062

In [56]:
train_cindex['Randomsurvivalforest'] = np.round(study_cindex.best_value, 3)
train_ibs['Randomsurvivalforest'] = np.round(study_ibs.best_value, 3)

In [57]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.847
train_ibs:  0.203


#### Test

In [58]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))
    
y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [59]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(RandomSurvivalForest, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex: ", c_index)

# Set the best model 
best_model_ibs = create_best_model(RandomSurvivalForest, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

RandomSurvivalForest(max_depth=12, max_features=None, max_leaf_nodes=9,
                     max_samples=0.7321959730404333, min_samples_leaf=2,
                     min_samples_split=3,
                     min_weight_fraction_leaf=0.011048621886878659,
                     n_estimators=70, random_state=123, warm_start=True)

test_cindex:  0.613


RandomSurvivalForest(max_depth=6, max_leaf_nodes=13,
                     max_samples=0.6084827037288676, min_samples_leaf=2,
                     min_samples_split=3,
                     min_weight_fraction_leaf=0.03342457164243477,
                     n_estimators=397, oob_score=True, random_state=123)

test_ibs:  0.219


In [60]:
# Saving the values to the dictionary 
test_cindex['Randomsurvivalforest'] = c_index
test_ibs['Randomsurvivalforest'] = ibs

### 6. ExtraSurvivalTrees

#### Train

In [61]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

In [62]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters 
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        warm_start = trial.suggest_categorical("warm_start", [True, False])
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics

        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score, 
                            max_features=max_features, 
                            warm_start=warm_start, 
                            max_samples=max_samples,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_depth=max_depth, 
                            random_state=123) 
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ExtraSurvivalTrees, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ExtraSurvivalTrees, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 02:06:38,160] A new study created in memory with name: no-name-39a1d3d3-8211-4344-9c73-bc3c87220427


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6060606060606061
Fold 2 C-index: 0.8080357142857143
Fold 3 C-index: 0.6372549019607843
Fold 4 C-index: 0.7172995780590717
Fold 5 C-index: 0.6713615023474179
[I 2024-04-16 02:06:40,044] Trial 0 finished with value: 0.6880024605427189 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7641958651588321, 'min_weight_fraction_leaf': 0.09124586522674999}. Best is trial 0 with value: 0.6880024605427189.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 02:06:44,692] Trial 1 finished with value: 0.5 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 12, 'min_samples_leaf': 11, 'max_depth': 13, 'n_estimators': 425, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.4877764869966794, 'min_weight_fraction_leaf': 0.2468425488251531

Fold 4 C-index: 0.630801687763713
Fold 5 C-index: 0.596244131455399
[I 2024-04-16 02:07:33,051] Trial 15 finished with value: 0.6572996018372016 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 9, 'min_samples_leaf': 13, 'max_depth': 4, 'n_estimators': 258, 'oob_score': False, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0.6880212408103346, 'min_weight_fraction_leaf': 0.07884420972256234}. Best is trial 12 with value: 0.7240013613915719.
Fold 1 C-index: 0.6493506493506493
Fold 2 C-index: 0.8303571428571429
Fold 3 C-index: 0.6053921568627451
Fold 4 C-index: 0.6244725738396625
Fold 5 C-index: 0.6032863849765259
[I 2024-04-16 02:07:35,487] Trial 16 finished with value: 0.6625717815773451 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 17, 'min_samples_leaf': 5, 'max_depth': 20, 'n_estimators': 499, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.8750346354626457, 'min_weight_fraction_leaf': 0.4093818399278544}. Best is 

Fold 1 C-index: 0.5930735930735931
Fold 2 C-index: 0.84375
Fold 3 C-index: 0.7254901960784313
Fold 4 C-index: 0.7721518987341772
Fold 5 C-index: 0.7511737089201878
[I 2024-04-16 02:08:13,663] Trial 30 finished with value: 0.7371278793612779 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 5, 'min_samples_leaf': 2, 'max_depth': 9, 'n_estimators': 278, 'oob_score': False, 'warm_start': True, 'max_features': None, 'max_samples': 0.7262614295600427, 'min_weight_fraction_leaf': 0.05879574792157449}. Best is trial 30 with value: 0.7371278793612779.
Fold 1 C-index: 0.5974025974025974
Fold 2 C-index: 0.8392857142857143
Fold 3 C-index: 0.7107843137254902
Fold 4 C-index: 0.759493670886076
Fold 5 C-index: 0.7136150234741784
[I 2024-04-16 02:08:15,173] Trial 31 finished with value: 0.7241162639548113 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 5, 'min_samples_leaf': 2, 'max_depth': 9, 'n_estimators': 281, 'oob_score': False, 'warm_start': True, 'max_features': None, 'max_s

...

Fold 1 IBS: 0.2079372669531199
Fold 2 IBS: 0.21242404225545655
Fold 3 IBS: 0.20045543323040851
Fold 4 IBS: 0.21635030239367528
Fold 5 IBS: 0.2147192598847946
[I 2024-04-16 02:14:30,037] Trial 30 finished with value: 0.210377260943491 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 2, 'min_samples_leaf': 2, 'max_depth': 16, 'n_estimators': 283, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.9203253020321057, 'min_weight_fraction_leaf': 0.10064443687249378}. Best is trial 24 with value: 0.20307708233862587.
Fold 1 IBS: 0.19727813448146547
Fold 2 IBS: 0.20858268136627384
Fold 3 IBS: 0.19133027071099074
Fold 4 IBS: 0.20961796894193035
Fold 5 IBS: 0.2168411172174772
[I 2024-04-16 02:14:37,816] Trial 31 finished with value: 0.2047300345436275 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 3, 'min_samples_leaf': 1, 'max_depth': 17, 'n_estimators': 287, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.837668

Fold 1 IBS: 0.21211475811792724
Fold 2 IBS: 0.21925589796211187
Fold 3 IBS: 0.20366334205091433
Fold 4 IBS: 0.2230512764619915
Fold 5 IBS: 0.21727676286318043
[I 2024-04-16 02:16:31,582] Trial 45 finished with value: 0.21507240749122508 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 4, 'min_samples_leaf': 10, 'max_depth': 14, 'n_estimators': 469, 'oob_score': True, 'warm_start': False, 'max_features': 0.1, 'max_samples': 0.7672440754515626, 'min_weight_fraction_leaf': 0.018436681014022648}. Best is trial 35 with value: 0.20216473764514423.
Fold 1 IBS: 0.2122142774753628
Fold 2 IBS: 0.2180461080401636
Fold 3 IBS: 0.20407939074493567
Fold 4 IBS: 0.22242943441636348
Fold 5 IBS: 0.2162867488284483
[I 2024-04-16 02:16:40,820] Trial 46 finished with value: 0.2146111919010548 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 7, 'min_samples_leaf': 6, 'max_depth': 13, 'n_estimators': 368, 'oob_score': True, 'warm_start': False, 'max_features': 1, 'max_samples': 0.3746418

Fold 1 IBS: 0.20305281534009662
Fold 2 IBS: 0.20585495766845885
Fold 3 IBS: 0.1955808964896082
Fold 4 IBS: 0.2074808706865538
Fold 5 IBS: 0.21451345880907943
[I 2024-04-16 02:18:17,678] Trial 60 finished with value: 0.2052965997987594 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 4, 'min_samples_leaf': 5, 'max_depth': 16, 'n_estimators': 417, 'oob_score': False, 'warm_start': True, 'max_features': None, 'max_samples': 0.7842199588841721, 'min_weight_fraction_leaf': 0.04700788452182417}. Best is trial 35 with value: 0.20216473764514423.
Fold 1 IBS: 0.1998780751185581
Fold 2 IBS: 0.20385564370285802
Fold 3 IBS: 0.19628236070807806
Fold 4 IBS: 0.20757349207763956
Fold 5 IBS: 0.2129679416601216
[I 2024-04-16 02:18:26,297] Trial 61 finished with value: 0.20411150265345107 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 20, 'min_samples_leaf': 4, 'max_depth': 15, 'n_estimators': 494, 'oob_score': False, 'warm_start': True, 'max_features': None, 'max_samples': 0.532147

Fold 1 IBS: 0.2077943851976713
Fold 2 IBS: 0.2106473346735518
Fold 3 IBS: 0.19934131700455673
Fold 4 IBS: 0.2159624844050163
Fold 5 IBS: 0.21457731226181645
[I 2024-04-16 02:20:20,438] Trial 75 finished with value: 0.20966456670852254 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 17, 'min_samples_leaf': 3, 'max_depth': 5, 'n_estimators': 444, 'oob_score': False, 'warm_start': True, 'max_features': None, 'max_samples': 0.4990384010791153, 'min_weight_fraction_leaf': 0.08585602005375931}. Best is trial 35 with value: 0.20216473764514423.
Fold 1 IBS: 0.20827922933803078
Fold 2 IBS: 0.21779573671850924
Fold 3 IBS: 0.20254172574221596
Fold 4 IBS: 0.219528369978886
Fold 5 IBS: 0.21421783307752998
[I 2024-04-16 02:20:28,766] Trial 76 finished with value: 0.2124725789710344 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 16, 'min_samples_leaf': 1, 'max_depth': 7, 'n_estimators': 499, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.330798

Fold 1 IBS: 0.2111555446843108
Fold 2 IBS: 0.2169096073289264
Fold 3 IBS: 0.20194158452124114
Fold 4 IBS: 0.2208206600975687
Fold 5 IBS: 0.2149901028297518
[I 2024-04-16 02:22:19,223] Trial 90 finished with value: 0.21316349989235978 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 18, 'min_samples_leaf': 7, 'max_depth': 5, 'n_estimators': 456, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.5525058622907164, 'min_weight_fraction_leaf': 0.014172537425326079}. Best is trial 83 with value: 0.20127111815945367.
Fold 1 IBS: 0.1992255839628862
Fold 2 IBS: 0.20148460763610068
Fold 3 IBS: 0.1931898251383408
Fold 4 IBS: 0.20591756041570602
Fold 5 IBS: 0.2141239831125513
[I 2024-04-16 02:22:27,960] Trial 91 finished with value: 0.202788312053117 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 20, 'min_samples_leaf': 3, 'max_depth': 8, 'n_estimators': 469, 'oob_score': False, 'warm_start': True, 'max_features': None, 'max_samples': 0.64928017

In [63]:
train_cindex['ExtraSurvivalTrees'] = np.round(study_cindex.best_value, 3)
train_ibs['ExtraSurvivalTrees'] = np.round(study_ibs.best_value, 3)

In [64]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.847
train_ibs:  0.201


#### Test

In [65]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [66]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ExtraSurvivalTrees, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ExtraSurvivalTrees, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ExtraSurvivalTrees(max_depth=16, max_features=None, max_leaf_nodes=16,
                   max_samples=0.9069233738536542, min_samples_leaf=2,
                   min_samples_split=3,
                   min_weight_fraction_leaf=0.010939301482941477,
                   n_estimators=470, random_state=123, warm_start=True)

C-index score: 0.618


ExtraSurvivalTrees(max_depth=7, max_features=None, max_leaf_nodes=18,
                   max_samples=0.6410631396472465, min_samples_split=2,
                   min_weight_fraction_leaf=0.016675107870370562,
                   n_estimators=481, random_state=123, warm_start=True)

IBS: 0.212


In [67]:
# Saving the values to the dictionary 
test_cindex['ExtraSurvivalTrees'] = c_index
test_ibs['ExtraSurvivalTrees'] = ibs

### 7. GradientBoostingSurvivalAnalysis

#### Train

In [68]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        criterion = trial.suggest_categorical('criterion', ['friedman_mse', 'squared_error'])
        ccp_alpha = trial.suggest_float("ccp_alpha", 0.0, 10)
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        min_impurity_decrease = trial.suggest_loguniform('min_impurity_decrease', 1e-7, 1e-1)
        validation_fraction = trial.suggest_float("validation_fraction", 0.0, 1.0)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            learning_rate=learning_rate,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            ccp_alpha=ccp_alpha, 
                            criterion=criterion,
                            min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            min_weight_fraction_leaf=min_weight_fraction_leaf,
                            max_depth=max_depth,
                            max_features=max_features,
                            max_leaf_nodes=max_leaf_nodes, 
                            min_impurity_decrease=min_impurity_decrease,
                            validation_fraction=validation_fraction, 
                            random_state=123)
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(GradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(GradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-16 02:23:37,674] A new study created in memory with name: no-name-fcb5e641-f6d9-4667-8038-2cd304daca0f


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 02:24:13,245] Trial 0 finished with value: 0.5 and parameters: {'subsample': 0.7268222670380755, 'learning_rate': 0.02932779416008757, 'dropout_rate': 0.3041663082077828, 'n_estimators': 276, 'criterion': 'friedman_mse', 'ccp_alpha': 9.807641983846155, 'min_weight_fraction_leaf': 0.34241486929243165, 'max_features': None, 'min_impurity_decrease': 2.4449249473284515e-05, 'validation_fraction': 0.7379954057320357, 'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 11, 'max_depth': 11}. Best is trial 0 with value: 0.5.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 02:24:31,613] Trial 1 finished with value: 0.5 and parameters: {'subsample': 0.6709608626961889, 'learning_rate': 0.08509374761370117, 'dropout_rate': 0.7520097923745717, 'n_estimators': 306, 'criterion': 'friedman_mse', 'ccp_alpha': 3.

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 02:32:04,110] Trial 13 finished with value: 0.5 and parameters: {'subsample': 0.7784830438231676, 'learning_rate': 0.02439641670004065, 'dropout_rate': 0.4821375662037144, 'n_estimators': 148, 'criterion': 'friedman_mse', 'ccp_alpha': 6.245821408640139, 'min_weight_fraction_leaf': 0.3469021849356369, 'max_features': 1, 'min_impurity_decrease': 0.002901900339595834, 'validation_fraction': 0.40492470595401014, 'min_samples_split': 6, 'max_leaf_nodes': 15, 'min_samples_leaf': 12, 'max_depth': 5}. Best is trial 9 with value: 0.6608490969281925.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 02:33:15,472] Trial 14 finished with value: 0.5 and parameters: {'subsample': 0.5522283925781899, 'learning_rate': 0.010913227078192221, 'dropout_rate': 0.275468298707492, 'n_estimators': 392, 'criterion': 'squared_error', 'c

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 02:40:23,096] Trial 26 finished with value: 0.5 and parameters: {'subsample': 0.8806073308147355, 'learning_rate': 0.022376976197093057, 'dropout_rate': 0.6881130985931054, 'n_estimators': 424, 'criterion': 'squared_error', 'ccp_alpha': 2.7801390404743485, 'min_weight_fraction_leaf': 0.14826348238129222, 'max_features': 1, 'min_impurity_decrease': 3.2746646321179855e-06, 'validation_fraction': 0.654336976255422, 'min_samples_split': 2, 'max_leaf_nodes': 18, 'min_samples_leaf': 14, 'max_depth': 5}. Best is trial 9 with value: 0.6608490969281925.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 02:41:22,937] Trial 27 finished with value: 0.5 and parameters: {'subsample': 0.6416570269538786, 'learning_rate': 0.08875752985571464, 'dropout_rate': 0.19627515205758927, 'n_estimators': 344, 'criterion': 'friedman_mse'

...

Fold 1 IBS: 0.21397652195149616
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.2181243152560957
[I 2024-04-16 03:36:27,592] Trial 45 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.5303064017076159, 'learning_rate': 0.013591809817627606, 'dropout_rate': 0.4506745791740544, 'n_estimators': 401, 'criterion': 'friedman_mse', 'ccp_alpha': 2.5557833119257145, 'min_weight_fraction_leaf': 0.3344828403640176, 'max_features': 0.1, 'min_impurity_decrease': 0.02254931450703544, 'validation_fraction': 0.5042239156018957, 'min_samples_split': 5, 'max_leaf_nodes': 12, 'min_samples_leaf': 15, 'max_depth': 14}. Best is trial 9 with value: 0.2165031446823389.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609563
[I 2024-04-16 03:36:57,635] Trial 46 finished with value: 0.21659054862241586 and parameters: {'subsam

Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609563
[I 2024-04-16 03:40:29,344] Trial 56 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.1173095704794419, 'learning_rate': 0.022797925457682687, 'dropout_rate': 0.2500192595896841, 'n_estimators': 209, 'criterion': 'squared_error', 'ccp_alpha': 1.515994978548666, 'min_weight_fraction_leaf': 0.04557038009296144, 'max_features': 'sqrt', 'min_impurity_decrease': 8.288990661784225e-06, 'validation_fraction': 0.6682715637714014, 'min_samples_split': 13, 'max_leaf_nodes': 19, 'min_samples_leaf': 4, 'max_depth': 7}. Best is trial 9 with value: 0.2165031446823389.
Fold 1 IBS: 0.2138266015910569
Fold 2 IBS: 0.2215272504056153
Fold 3 IBS: 0.20423591913518002
Fold 4 IBS: 0.22463204530996436
Fold 5 IBS: 0.21804570777294438
[I 2024-04-16 03:40:45,904] Trial 57 finished with value: 0.2164535048429522 and parameters: {'subsample': 0.553868700002313, 'lea

Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609563
[I 2024-04-16 03:46:55,967] Trial 67 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.44533772449036413, 'learning_rate': 0.0814448088163274, 'dropout_rate': 0.8313520760085876, 'n_estimators': 440, 'criterion': 'friedman_mse', 'ccp_alpha': 0.5496755232313972, 'min_weight_fraction_leaf': 0.23846073397265538, 'max_features': 'log2', 'min_impurity_decrease': 0.0245829172127866, 'validation_fraction': 0.8751349059723512, 'min_samples_split': 17, 'max_leaf_nodes': 13, 'min_samples_leaf': 12, 'max_depth': 4}. Best is trial 63 with value: 0.21622350745255856.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609574
[I 2024-04-16 03:47:33,837] Trial 68 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.49945879365729917, 

Fold 2 IBS: 0.22103795223774314
Fold 3 IBS: 0.20401739251272527
Fold 4 IBS: 0.22443280508876015
Fold 5 IBS: 0.21789254751435055
[I 2024-04-16 03:52:27,130] Trial 78 finished with value: 0.21620496495206992 and parameters: {'subsample': 0.3352733938121141, 'learning_rate': 0.07411999177509665, 'dropout_rate': 0.8605229004531987, 'n_estimators': 473, 'criterion': 'squared_error', 'ccp_alpha': 0.023234528924181166, 'min_weight_fraction_leaf': 0.20870378808533252, 'max_features': 'log2', 'min_impurity_decrease': 0.007686471270079544, 'validation_fraction': 0.6325076348170467, 'min_samples_split': 19, 'max_leaf_nodes': 14, 'min_samples_leaf': 11, 'max_depth': 5}. Best is trial 75 with value: 0.21603046229391182.
Fold 1 IBS: 0.21367567354444209
Fold 2 IBS: 0.22106789416004893
Fold 3 IBS: 0.20416799678888137
Fold 4 IBS: 0.22445627718920175
Fold 5 IBS: 0.2178911936966356
[I 2024-04-16 03:52:38,730] Trial 79 finished with value: 0.21625180707584196 and parameters: {'subsample': 0.33166032751272

Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609566
[I 2024-04-16 03:55:22,520] Trial 89 finished with value: 0.21659054862241583 and parameters: {'subsample': 0.40845912170347753, 'learning_rate': 0.07409455257113193, 'dropout_rate': 0.9031580025226588, 'n_estimators': 489, 'criterion': 'squared_error', 'ccp_alpha': 1.426169496743956, 'min_weight_fraction_leaf': 0.17670307432796536, 'max_features': 'log2', 'min_impurity_decrease': 0.001833888378396466, 'validation_fraction': 0.7135149436193794, 'min_samples_split': 20, 'max_leaf_nodes': 13, 'min_samples_leaf': 13, 'max_depth': 3}. Best is trial 75 with value: 0.21603046229391182.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.22473737834741545
Fold 5 IBS: 0.21812431525609557
[I 2024-04-16 03:55:42,259] Trial 90 finished with value: 0.21659041601237358 and parameters: {'subsample': 0.2452119153654136

In [69]:
train_cindex['GradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['GradientBoosting'] = np.round(study_ibs.best_value, 3)

In [70]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.661
train_ibs:  0.216


#### Test

In [71]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [72]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(GradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(GradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

GradientBoostingSurvivalAnalysis(ccp_alpha=0.07426378544613033,
                                 criterion='squared_error',
                                 dropout_rate=0.28125955124323376,
                                 learning_rate=0.013102111413618,
                                 max_features='auto', max_leaf_nodes=16,
                                 min_impurity_decrease=1.4994028685178666e-07,
                                 min_samples_leaf=10,
                                 min_weight_fraction_leaf=0.27579636299120275,
                                 n_estimators=406, random_state=123,
                                 subsample=0.6059965408578151,
                                 validation_fraction=0.6359003593513561)

C-index score: 0.617


GradientBoostingSurvivalAnalysis(ccp_alpha=0.009365616677551508,
                                 dropout_rate=0.7826693566389662,
                                 learning_rate=0.08843409287484169, max_depth=4,
                                 max_features='log2', max_leaf_nodes=14,
                                 min_impurity_decrease=0.0008096076350961149,
                                 min_samples_leaf=16, min_samples_split=19,
                                 min_weight_fraction_leaf=0.2812508412156809,
                                 n_estimators=403, random_state=123,
                                 subsample=0.5200735929114648,
                                 validation_fraction=0.7004324115891782)

IBS: 0.221


In [73]:
# Saving the values to the dictionary 
test_cindex['GradientBoosting'] = c_index
test_ibs['GradientBoosting'] = ibs

### 8. ComponentwiseGradientBoostingSurvivalAnalysis

#### Train

In [74]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

In [75]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            learning_rate=learning_rate,
                            random_state=123)
                
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective


# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best hyperparameters for C-index: \n", study_cindex.best_params)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best hyperparameters for IBS: \n", study_ibs.best_params)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 03:59:16,739] A new study created in memory with name: no-name-1673607a-f872-4908-9c63-8b9aa056f559


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6017316017316018
Fold 2 C-index: 0.7098214285714286
Fold 3 C-index: 0.7156862745098039
Fold 4 C-index: 0.7637130801687764
Fold 5 C-index: 0.6244131455399061
[I 2024-04-16 03:59:18,262] Trial 0 finished with value: 0.6830731061043034 and parameters: {'subsample': 0.7268222670380755, 'dropout_rate': 0.3575254014553415, 'n_estimators': 114, 'learning_rate': 0.05558016213920623}. Best is trial 0 with value: 0.6830731061043034.
Fold 1 C-index: 0.6017316017316018
Fold 2 C-index: 0.7098214285714286
Fold 3 C-index: 0.7156862745098039
Fold 4 C-index: 0.7637130801687764
Fold 5 C-index: 0.6244131455399061
[I 2024-04-16 03:59:27,908] Trial 1 finished with value: 0.6830731061043034 and parameters: {'subsample': 0.7475220728070068, 'dropout_rate': 0.4807958141120149, 'n_estimators': 491, 'learning_rate': 0.06879814411990147}. Best is trial 0 with value: 0.6830731061043034.
Fold 1 C-index: 0.6017316017316018
Fold 2 C-index: 0.7098214285714286
Fold 3 C-index: 0.7156862745098039
Fold 

Fold 1 C-index: 0.6017316017316018
Fold 2 C-index: 0.7098214285714286
Fold 3 C-index: 0.7156862745098039
Fold 4 C-index: 0.7721518987341772
Fold 5 C-index: 0.6244131455399061
[I 2024-04-16 04:00:40,069] Trial 19 finished with value: 0.6847608698173835 and parameters: {'subsample': 0.765021860726683, 'dropout_rate': 0.28451388926437, 'n_estimators': 299, 'learning_rate': 0.009486725203279353}. Best is trial 6 with value: 0.6847608698173835.
Fold 1 C-index: 0.6017316017316018
Fold 2 C-index: 0.7098214285714286
Fold 3 C-index: 0.7156862745098039
Fold 4 C-index: 0.7637130801687764
Fold 5 C-index: 0.6244131455399061
[I 2024-04-16 04:00:46,870] Trial 20 finished with value: 0.6830731061043034 and parameters: {'subsample': 0.6102628884762309, 'dropout_rate': 0.5936662553833888, 'n_estimators': 386, 'learning_rate': 0.039803505489267865}. Best is trial 6 with value: 0.6847608698173835.
Fold 1 C-index: 0.6017316017316018
Fold 2 C-index: 0.7098214285714286
Fold 3 C-index: 0.7156862745098039
Fold

Fold 1 C-index: 0.6017316017316018
Fold 2 C-index: 0.7098214285714286
Fold 3 C-index: 0.7156862745098039
Fold 4 C-index: 0.7721518987341772
Fold 5 C-index: 0.6244131455399061
[I 2024-04-16 04:02:10,367] Trial 38 finished with value: 0.6847608698173835 and parameters: {'subsample': 0.7933691320344304, 'dropout_rate': 0.3000960517027121, 'n_estimators': 356, 'learning_rate': 0.013887612982787472}. Best is trial 6 with value: 0.6847608698173835.
Fold 1 C-index: 0.6017316017316018
Fold 2 C-index: 0.71875
Fold 3 C-index: 0.7205882352941176
Fold 4 C-index: 0.7552742616033755
Fold 5 C-index: 0.6244131455399061
[I 2024-04-16 04:02:18,845] Trial 39 finished with value: 0.6841514488338002 and parameters: {'subsample': 0.3246512531312218, 'dropout_rate': 0.4491392168970607, 'n_estimators': 388, 'learning_rate': 0.04653270707709469}. Best is trial 6 with value: 0.6847608698173835.
Fold 1 C-index: 0.6017316017316018
Fold 2 C-index: 0.7098214285714286
Fold 3 C-index: 0.7156862745098039
Fold 4 C-inde

...

Fold 4 IBS: 0.2080662242520272
Fold 5 IBS: 0.228727148642183
[I 2024-04-16 04:06:31,166] Trial 19 finished with value: 0.20234080958023443 and parameters: {'subsample': 0.10013306718236009, 'dropout_rate': 0.9930703343219778, 'n_estimators': 51, 'learning_rate': 0.04096883532946153}. Best is trial 7 with value: 0.2005497416666168.
Fold 1 IBS: 0.19900268568861765
Fold 2 IBS: 0.19673124531818784
Fold 3 IBS: 0.18097673731541652
Fold 4 IBS: 0.20623445559536285
Fold 5 IBS: 0.22394152158247504
[I 2024-04-16 04:06:31,904] Trial 20 finished with value: 0.20137732910001196 and parameters: {'subsample': 0.1435040576337751, 'dropout_rate': 0.7933084651006226, 'n_estimators': 43, 'learning_rate': 0.04127909023805986}. Best is trial 7 with value: 0.2005497416666168.
Fold 1 IBS: 0.20102182120527576
Fold 2 IBS: 0.20177601792835187
Fold 3 IBS: 0.1880831686670229
Fold 4 IBS: 0.21737450486532756
Fold 5 IBS: 0.22131729031914832
[I 2024-04-16 04:06:32,593] Trial 21 finished with value: 0.20591456059702526

Fold 5 IBS: 0.22465026642731892
[I 2024-04-16 04:06:56,964] Trial 38 finished with value: 0.2023805862045262 and parameters: {'subsample': 0.17017303600682596, 'dropout_rate': 0.6744028301161138, 'n_estimators': 84, 'learning_rate': 0.02105811333204098}. Best is trial 36 with value: 0.20053173745597602.
Fold 1 IBS: 0.2293570713772131
Fold 2 IBS: 0.21090831309405633
Fold 3 IBS: 0.1854937460956396
Fold 4 IBS: 0.2135188317245189
Fold 5 IBS: 0.3310658823973223
[I 2024-04-16 04:06:59,215] Trial 39 finished with value: 0.23406876893775003 and parameters: {'subsample': 0.3439557983328422, 'dropout_rate': 0.844167451106476, 'n_estimators': 194, 'learning_rate': 0.051973238153326454}. Best is trial 36 with value: 0.20053173745597602.
Fold 1 IBS: 0.21206377067603904
Fold 2 IBS: 0.1864274650720211
Fold 3 IBS: 0.17644966324602238
Fold 4 IBS: 0.19294957610953867
Fold 5 IBS: 0.2664281407354722
[I 2024-04-16 04:07:00,751] Trial 40 finished with value: 0.20686372316781867 and parameters: {'subsample':

Fold 5 IBS: 0.22884938440987548
[I 2024-04-16 04:07:46,837] Trial 57 finished with value: 0.20081171638871226 and parameters: {'subsample': 0.31114891339062667, 'dropout_rate': 0.6489378315882792, 'n_estimators': 251, 'learning_rate': 0.007801704420672669}. Best is trial 50 with value: 0.2002407140843617.
Fold 1 IBS: 0.2180531959388824
Fold 2 IBS: 0.1850434781595872
Fold 3 IBS: 0.18125157795710164
Fold 4 IBS: 0.1957099312923242
Fold 5 IBS: 0.2808496755196348
[I 2024-04-16 04:07:49,240] Trial 58 finished with value: 0.21218157177350605 and parameters: {'subsample': 0.3221818826721476, 'dropout_rate': 0.5833347466815137, 'n_estimators': 202, 'learning_rate': 0.027679523779857756}. Best is trial 50 with value: 0.2002407140843617.
Fold 1 IBS: 0.21120995027458148
Fold 2 IBS: 0.1838676821243201
Fold 3 IBS: 0.17761827785583964
Fold 4 IBS: 0.1945128293118185
Fold 5 IBS: 0.2635788119779368
[I 2024-04-16 04:07:52,935] Trial 59 finished with value: 0.20615751030889928 and parameters: {'subsample'

Fold 5 IBS: 0.29644781879545157
[I 2024-04-16 04:08:46,278] Trial 76 finished with value: 0.2186462067577791 and parameters: {'subsample': 0.2534219458429501, 'dropout_rate': 0.619685631002236, 'n_estimators': 236, 'learning_rate': 0.029611588065970945}. Best is trial 50 with value: 0.2002407140843617.
Fold 1 IBS: 0.19966889046600855
Fold 2 IBS: 0.19925346497055635
Fold 3 IBS: 0.181090164247083
Fold 4 IBS: 0.20616955499619105
Fold 5 IBS: 0.224445533329019
[I 2024-04-16 04:08:47,521] Trial 77 finished with value: 0.20212552160177158 and parameters: {'subsample': 0.3885243355394725, 'dropout_rate': 0.7696332658571577, 'n_estimators': 61, 'learning_rate': 0.025174387951611213}. Best is trial 50 with value: 0.2002407140843617.
Fold 1 IBS: 0.19976721983416995
Fold 2 IBS: 0.20024051454876698
Fold 3 IBS: 0.18479863559918963
Fold 4 IBS: 0.21154594555854472
Fold 5 IBS: 0.22205446508890664
[I 2024-04-16 04:08:49,001] Trial 78 finished with value: 0.2036813561259156 and parameters: {'subsample': 

Fold 4 IBS: 0.20119576053683905
Fold 5 IBS: 0.22983373825943978
[I 2024-04-16 04:10:42,992] Trial 95 finished with value: 0.20030028617935028 and parameters: {'subsample': 0.4757995705798828, 'dropout_rate': 0.6231094767303963, 'n_estimators': 479, 'learning_rate': 0.004160712641332346}. Best is trial 92 with value: 0.19989779094085552.
Fold 1 IBS: 0.19933033684879545
Fold 2 IBS: 0.19175290459754746
Fold 3 IBS: 0.17523641758102454
Fold 4 IBS: 0.19898064100110172
Fold 5 IBS: 0.23416739031725986
[I 2024-04-16 04:10:51,266] Trial 96 finished with value: 0.19989353806914578 and parameters: {'subsample': 0.4540175560315793, 'dropout_rate': 0.5727605083055514, 'n_estimators': 452, 'learning_rate': 0.005218799046830834}. Best is trial 96 with value: 0.19989353806914578.
Fold 1 IBS: 0.2016948797357464
Fold 2 IBS: 0.20332294880280635
Fold 3 IBS: 0.18477449173738422
Fold 4 IBS: 0.20962380699873429
Fold 5 IBS: 0.2218130841298247
[I 2024-04-16 04:10:59,566] Trial 97 finished with value: 0.20424584

In [76]:
train_cindex['ComponentwiseGradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['ComponentwiseGradientBoosting'] = np.round(study_ibs.best_value, 3)

In [77]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.688
train_ibs:  0.2


#### Test

In [78]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [79]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.7166562602631428,
                                              learning_rate=0.07736008213803257,
                                              n_estimators=227,
                                              random_state=123,
                                              subsample=0.3054811831292578)

C-index score: 0.634


ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.5727605083055514,
                                              learning_rate=0.005218799046830834,
                                              n_estimators=452,
                                              random_state=123,
                                              subsample=0.4540175560315793)

IBS: 0.206


In [80]:
# Saving the values to the dictionary 
test_cindex['ComponentwiseGradientBoosting'] = c_index
test_ibs['ComponentwiseGradientBoosting'] = ibs

# Results

In [81]:
df_train_cindex = pd.DataFrame(train_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_train_cindex['rank'] = df_train_cindex['C-index'].rank(ascending=False)
df_train_cindex 

,C-index,rank
Randomsurvivalforest,0.847,1.5
ExtraSurvivalTrees,0.847,1.5
ComponentwiseGradientBoosting,0.688,3.0
CoxRidge,0.675,4.0
GradientBoosting,0.661,5.0
CoxElastic,0.658,6.0
CoxLasso,0.657,7.0
CoxPH,0.656,8.0


In [82]:
df_train_ibs = pd.DataFrame(train_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_train_ibs['rank'] = df_train_ibs['IBS'].rank(ascending=True)
df_train_ibs

,IBS,rank
ComponentwiseGradientBoosting,0.200,1.0
ExtraSurvivalTrees,0.201,2.0
Randomsurvivalforest,0.203,3.0
CoxPH,0.206,5.0
CoxLasso,0.206,5.0
CoxElastic,0.206,5.0
GradientBoosting,0.216,7.0
CoxRidge,0.217,8.0


In [83]:
df_test_cindex = pd.DataFrame(test_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_test_cindex['rank'] = df_test_cindex['C-index'].rank(ascending=False)
df_test_cindex 

,C-index,rank
ComponentwiseGradientBoosting,0.634,1.0
ExtraSurvivalTrees,0.618,2.0
GradientBoosting,0.617,3.0
CoxRidge,0.616,4.0
Randomsurvivalforest,0.613,5.0
CoxPH,0.591,6.0
CoxLasso,0.590,7.5
CoxElastic,0.590,7.5


In [84]:
df_test_ibs = pd.DataFrame(test_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_test_ibs['rank'] = df_test_ibs['IBS'].rank(ascending=True)
df_test_ibs

,IBS,rank
ComponentwiseGradientBoosting,0.206,1.0
ExtraSurvivalTrees,0.212,2.0
Randomsurvivalforest,0.219,3.0
CoxRidge,0.221,4.5
GradientBoosting,0.221,4.5
CoxLasso,0.237,6.5
CoxElastic,0.237,6.5
CoxPH,0.238,8.0


In [85]:
# Renaming the column "index" to "model" 
df_train_cindex = df_train_cindex.reset_index().rename(columns={"index": "model"})
df_train_ibs = df_train_ibs.reset_index().rename(columns={"index": "model"})
df_test_cindex = df_test_cindex.reset_index().rename(columns={"index": "model"})
df_test_ibs = df_test_ibs.reset_index().rename(columns={"index": "model"})

# Save the files 
dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  
file_path = 'path_to_your_folder/'  

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']


dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  
file_path = '/Users/minjeongcheon/Desktop/results_thesis/d2/os/yeojohnson/plsr/' 

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']

# Modify the file names to match the desired format
modified_file_names = ['d2_os_yeojohnson_plsr_' + file_name for file_name in file_names]

# Loop through each DataFrame and save them with corresponding modified file names
for df, modified_file_name in zip(dfs, modified_file_names):
    file_path_name = file_path + modified_file_name  # Construct the full file path
    df.to_csv(file_path_name, index=False)  # Save the DataFrame to CSV file


In [86]:
from datetime import date

current_date = date.today()
print(current_date)

2024-04-16
